# 4장 실습 — 속도를 바꾸면 차량 일정이 어떻게 달라질까

하남시청에서 미사역으로 가는 시간을 자유류, 오전 첨두, 오후 첨두, 심야 속도로 계산합니다.
그중 자유류와 오후 첨두의 경로를 그림으로 비교하고, 통행시간 차이가 다음 배차 시각에 어떻게 이어지는지 확인합니다.
마지막에는 같은 출발지와 목적지를 `Dtumos.route`에 전달해 결과를 읽습니다.

[교재 4장](../ko/ch04_speeds_engine.md)을 읽은 뒤 위에서부터 실행합니다. 별도의 서버는 필요하지 않습니다.
함수 내부를 모두 구현할 필요는 없습니다. 각 셀에서 **무엇을 넣고, 어떤 결과를 받아, 어디에 쓰는지**를 따라가 봅시다.

## 준비

아래 셀은 노트북에서 교재의 파이썬 패키지를 찾고, 표와 그림을 출력할 준비를 합니다.
`pandas`는 표를, `matplotlib`은 그림을 다루는 도구입니다. 이 셀은 그대로 실행합니다.

In [ ]:
import sys  # 파이썬이 패키지를 찾는 폴더 목록을 다루는 도구입니다.
from pathlib import Path  # 현재 폴더와 상위 폴더의 경로를 다루는 도구입니다.

for p in (Path.cwd(), *Path.cwd().parents):  # 현재 폴더부터 상위 폴더를 차례로 살펴봅니다.
    if (p / "smartmob").is_dir():  # 교재 패키지 폴더가 있는 곳을 찾습니다.
        ROOT = p  # 찾은 폴더를 이 저장소의 시작 위치로 기억합니다.
        break  # 저장소를 찾았으므로 더 위의 폴더는 살펴보지 않습니다.
sys.path.insert(0, str(ROOT))  # 이 저장소의 패키지를 파이썬이 불러올 수 있게 합니다.

import pandas as pd  # 계산 결과를 행과 열이 있는 표로 정리합니다.
import matplotlib.pyplot as plt  # 경로 좌표를 그림으로 그립니다.
from IPython.display import display  # 노트북에 표를 읽기 좋게 출력합니다.
from smartmob.viz import use_korean_font  # 그림에서 한글이 깨지지 않도록 글꼴을 설정합니다.

use_korean_font()  # 현재 환경에서 사용할 수 있는 한글 글꼴을 적용합니다.

## 1. 자유류 통행시간을 기준으로 잡습니다

3장에서 쓴 하남시청→미사역을 다시 계산합니다. `origin`과 `destination`은 `(위도, 경도)` 순서입니다.
`load_road_graph`가 도로망과 엣지 비용을 준비하고, `shortest_path`가 그 위에서 경로를 찾습니다.
속도 컬럼을 따로 지정하지 않으면 자유류 속도를 사용합니다.

계산 결과에는 경로를 구성하는 노드 목록 `nodes`와 초 단위 통행시간 `duration_s`가 들어 있습니다.
여기서는 통행시간을 분으로 바꾸어 읽습니다.

In [ ]:
from smartmob.data import load_road_graph  # 도로망을 읽고 길이와 속도로 엣지 비용을 만드는 함수입니다.
from smartmob.teaching.dijkstra import shortest_path  # 두 좌표 사이의 경로와 통행시간을 구하는 함수입니다.

origin = (37.5393, 127.2148)  # 출발지인 하남시청의 위도와 경도입니다.
destination = (37.5606, 127.1930)  # 목적지인 미사역의 위도와 경도입니다.
g_free = load_road_graph("hanam", modes=("drive",))  # 하남시 자동차 도로망을 자유류 속도로 준비합니다.
p_free = shortest_path(g_free, origin, destination, algorithm="dijkstra")  # 다익스트라로 가장 빠른 경로를 구합니다.
print(f"자유류 통행시간: {p_free.duration_s / 60:.2f}분")  # 초를 60으로 나누고 소수 둘째 자리까지 표시합니다.

약 5.54분입니다. 이 값은 도로가 한산하다고 가정한 속도로 계산한 결과입니다.
오후 6시에도 같은 시간을 사용할 수 있을지, 먼저 도로별 속도 자료를 확인합니다.

## 2. 바꿔 넣을 속도 자료를 확인합니다

`g_free.edges`는 방금 읽은 자동차 도로의 엣지 표입니다.
`free_flow_speed_kmh`는 자유류 속도, `weekday_pm_peak_p50`은 주중 오후 첨두의 관측 속도 중앙값입니다.
`p50`은 관측값의 50번째 백분위수, 즉 중앙값을 뜻합니다.

두 컬럼의 속도 중앙값과 비어 있는 값의 비율을 구합니다.
여기서 출력하는 중앙값은 각 컬럼을 다시 도로 전체에 걸쳐 요약한 값입니다.

In [ ]:
edges = g_free.edges  # 이미 읽은 자동차 도로의 엣지 표를 사용합니다.
columns = ["free_flow_speed_kmh", "weekday_pm_peak_p50"]  # 자유류와 오후 첨두 속도만 비교합니다.
speed_stats = edges[columns].median().to_frame("중앙값_kmh")  # 각 컬럼의 중앙값을 구해 표로 만듭니다.
speed_stats["결측_비율"] = edges[columns].isna().mean()  # 값이 없는 행을 True로 표시한 뒤 평균을 내어 비율을 구합니다.
display(speed_stats.round(3))  # 두 속도 조건의 통계를 소수 셋째 자리까지 보여 줍니다.

자유류 중앙값은 30km/h, 오후 첨두 중앙값은 약 18.9km/h입니다.
이 값은 엣지별 속도를 요약한 것이며, 하남시 전체 차량의 평균 주행 속도는 아닙니다.

오후 첨두의 `결측_비율`은 약 0.217, 즉 21.7%입니다.
`load_road_graph`는 관측이 없는 엣지를 같은 도로의 자유류 속도로 채웁니다.
자유류 값도 없으면 30km/h를 사용합니다. 이후 계산에는 이 대체 규칙도 적용됩니다.

## 3. 속도를 바꿔 시간과 경로를 비교합니다

오전 8시, 오후 6시, 심야 23시의 관측 속도를 사용합니다.
`speed_columns`는 표에 표시할 이름과 실제 속도 컬럼을 연결하는 사전입니다.
새 시간대를 비교하고 싶으면 이 사전에 항목을 추가하면 됩니다.

In [ ]:
speed_columns = {}  # 비교할 시간대 이름과 속도 컬럼을 짝지어 담습니다.
speed_columns["오전 8시"] = "weekday_am_peak_p50"  # 오전 7시 이상 9시 미만에 해당하는 관측 속도입니다.
speed_columns["오후 6시"] = "weekday_pm_peak_p50"  # 오후 5시 이상 7시 미만에 해당하는 관측 속도입니다.
speed_columns["심야 23시"] = "weekday_night_p50"  # 밤 10시부터 다음 날 오전 6시 전까지의 관측 속도입니다.

이제 같은 도로망에 속도 컬럼을 하나씩 바꿔 넣고, 같은 출발지와 목적지 사이의 경로를 다시 구합니다.
예를 들어 `speed_column="weekday_pm_peak_p50"`이면 각 엣지의 오후 첨두 속도로 비용을 만듭니다.

`graphs`에는 조건별 도로망을, `paths`에는 그 도로망에서 구한 경로를 보관합니다.
각 경로 계산은 선택한 속도를 끝까지 고정합니다. 이동 중 시간대가 바뀌는 경우는 반영하지 않습니다.

In [ ]:
graphs = {"자유류": g_free}  # 뒤에서 경로를 그릴 수 있도록 자유류 도로망부터 보관합니다.
paths = {"자유류": p_free}  # 앞에서 구한 자유류 경로를 비교 기준으로 보관합니다.

for label, column in speed_columns.items():  # 시간대 이름과 그에 해당하는 속도 컬럼을 하나씩 꺼냅니다.
    graph = load_road_graph("hanam", speed_column=column)  # 선택한 속도로 같은 자동차 도로망의 비용을 계산합니다.
    path = shortest_path(graph, origin, destination, algorithm="dijkstra")  # 바뀐 비용으로 최단경로를 다시 찾습니다.
    graphs[label] = graph  # 시간대 이름으로 해당 도로망을 꺼낼 수 있게 저장합니다.
    paths[label] = path  # 같은 이름으로 경로와 통행시간을 꺼낼 수 있게 저장합니다.

`paths["오후 6시"]`를 읽으면 오후 첨두의 계산 결과를 얻을 수 있습니다.
아래 표에서는 조건별 통행시간과 자유류 경로가 유지되는지를 함께 봅니다.
노드 수가 같아도 다른 길일 수 있으므로 노드 목록 전체를 비교합니다.

In [ ]:
rows = []  # 속도 조건마다 결과 한 행을 담을 목록입니다.
for label, path in paths.items():  # 자유류와 세 시간대의 경로를 차례로 읽습니다.
    minutes = path.duration_s / 60  # 경로의 총통행시간을 초에서 분으로 바꿉니다.
    same_path = path.nodes == p_free.nodes  # 지나가는 노드와 순서가 자유류 경로와 모두 같은지 확인합니다.
    rows.append([label, minutes, same_path])  # 조건 이름, 통행시간, 경로 비교 결과를 한 행으로 묶습니다.
comparison = pd.DataFrame(rows, columns=["속도_조건", "통행시간_분", "자유류와_같은_경로"])  # 행 목록에 열 이름을 붙입니다.
comparison = comparison.set_index("속도_조건")  # 뒤에서 조건 이름으로 행을 선택할 수 있게 합니다.
display(comparison.round(2))  # 시간은 소수 둘째 자리까지 표시하고, 경로가 같으면 True로 보여 줍니다.

자유류는 약 5.54분, 오후 첨두는 약 8.99분으로 약 62% 차이입니다.
관측 속도를 쓴 세 조건 모두 `자유류와_같은_경로`가 `False`입니다.
입력한 속도가 바뀌면서 통행시간과 선택된 경로가 함께 달라졌습니다.

자유류와 오후 첨두 경로를 좌표 평면에 겹쳐 봅니다.
`coords`가 돌려주는 순서는 `(위도, 경도)`이므로, 가로축에 경도와 세로축에 위도를 놓습니다.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))  # 두 경로를 겹쳐 그릴 그림과 좌표축을 만듭니다.
for label in ["자유류", "오후 6시"]:  # 차이를 볼 두 조건만 선택합니다.
    coords = paths[label].coords(graphs[label])  # 경로의 노드 번호를 해당 도로망의 위도·경도로 바꿉니다.
    positions = pd.DataFrame(coords, columns=["위도", "경도"])  # 좌표를 위도 열과 경도 열로 나눕니다.
    ax.plot(positions["경도"], positions["위도"], label=label)  # 경로 순서대로 좌표를 선으로 연결합니다.
ax.scatter(origin[1], origin[0], color="black", label="하남시청", zorder=3)  # 출발점을 경로선 위에 표시합니다.
ax.scatter(destination[1], destination[0], color="red", label="미사역", zorder=3)  # 도착점을 경로선 위에 표시합니다.
ax.set_xlabel("경도")  # 가로축 좌표가 경도임을 표시합니다.
ax.set_ylabel("위도")  # 세로축 좌표가 위도임을 표시합니다.
ax.set_title("하남시청 → 미사역: 속도 조건에 따른 경로")  # 비교하는 구간과 조건을 제목에 적습니다.
ax.legend()  # 선과 점이 어떤 경로·지점을 뜻하는지 범례를 표시합니다.
fig.tight_layout()  # 제목과 축 이름이 그림 밖으로 잘리지 않도록 여백을 맞춥니다.
plt.show()  # 완성한 경로 비교 그림을 출력합니다.

두 경로는 출발지와 목적지가 같지만 중간에 지나는 노드가 다릅니다.
이 그림은 노드 사이를 직선으로 이은 것이어서 도로의 굽은 형상까지 모두 나타내지는 않습니다.
표의 시간 차이와 그림의 경로 차이는 이 구간의 결과입니다. 다른 구간에서도 같은 비율로 늘어난다고 볼 수는 없습니다.

## 4. 통행시간을 다음 배차 시각에 연결합니다

18시에 호출을 받은 차량이 승객에게 가는 데 4분, 승차와 하차에 각각 1분을 쓴다고 가정합니다.
승객을 태운 뒤 하남시청→미사역으로 가는 시간만 앞에서 구한 값으로 바꿉니다.

다음 배차까지 걸리는 시간은 `픽업 시간 + 승차 시간 + 승객 통행시간 + 하차 시간`입니다.
호출 13분 뒤인 18:13에 새 요청이 들어올 때, 이 차량이 비어 있는지 비교합니다.

In [ ]:
pickup_min = 4  # 차량이 승객을 태우러 가는 시간을 4분으로 고정합니다.
stop_min = 1 + 1  # 승차 1분과 하차 1분을 더합니다.
schedule = comparison.loc[["자유류", "오후 6시"], ["통행시간_분"]].copy()  # 비교할 두 조건의 통행시간을 가져옵니다.
schedule["다음_배차까지_분"] = pickup_min + stop_min + schedule["통행시간_분"]  # 호출부터 하차 완료까지의 시간을 더합니다.
schedule["13분_뒤_배차가능"] = schedule["다음_배차까지_분"] <= 13  # 18:13까지 하차를 마쳤으면 True입니다.
display(schedule.round(2))  # 통행시간 가정에 따라 차량의 가용 상태가 달라지는지 확인합니다.

자유류에서는 호출 약 11.54분 뒤, 오후 첨두에서는 약 14.99분 뒤에 다음 배차가 가능합니다.
시각으로는 각각 약 18:11:32와 18:14:59입니다. 18:13의 호출에는 자유류 조건에서만 이 차량을 보낼 수 있습니다.
통행시간을 짧게 잡으면 같은 차량을 실제보다 일찍 다시 쓸 수 있다고 계산할 수 있습니다.

이는 픽업 시간을 고정한 차량 한 대의 예입니다.
전체 승객의 대기시간을 계산하려면 다른 차량과 새 호출까지 함께 처리해야 하며, 그 과정은 10~11장에서 다룹니다.

## 5. 같은 구간을 DTUMOS 호출로 확인합니다

앞에서는 도로망을 직접 읽고 `shortest_path`에 전달했습니다.
`Dtumos.route`를 사용할 때는 도시 이름과 출발·도착 좌표를 전달하고 결과를 받습니다.
여기서는 서버 없이 실행하도록 `mode="local"`을 지정합니다.
저장된 응답이 있으면 그것을 읽고, 없으면 내장 다익스트라가 경로를 계산합니다.

응답의 `duration`은 초, `distance`는 미터입니다.
`route`는 `[경도, 위도]` 순서의 좌표 목록이며, 입력 좌표의 순서와 반대입니다.

In [ ]:
from smartmob import Dtumos  # 교재에서 사용하는 DTUMOS 호출 클래스를 불러옵니다.

dt = Dtumos(mode="local")  # 서버에 접속하지 않고 실행할 호출 객체를 만듭니다.
result = dt.route("hanam", origin=origin, destination=destination)  # 같은 두 지점 사이의 경로를 요청합니다.
print(f"통행시간: {result['duration'] / 60:.2f}분")  # 응답의 초 단위 시간을 분으로 바꾸어 읽습니다.
print(f"거리: {result['distance'] / 1000:.2f}km")  # 응답의 미터 단위 거리를 km로 바꾸어 읽습니다.
print("처음 두 경로 좌표:", result["route"][:2])  # 경로가 [경도, 위도] 좌표 목록으로 들어 있는지 확인합니다.

이 구간의 기본 로컬 계산은 약 5.54분입니다.
이 호출에는 출발 시각이나 속도 컬럼을 전달하지 않았으므로 오후 첨두 결과로 해석하지 않습니다.
실서버와 비교할 때도 서버가 사용하는 도로망과 속도 조건을 먼저 확인합니다.

이 노트북에서 속도 컬럼을 바꾸는 것은 경로 계산의 조건을 바꾸는 일입니다.
`run_simulation`의 통행시간 설정에 자동으로 반영되지는 않습니다.

## 직접 바꾸어 보기

3절의 `speed_columns`에 저녁 8시 조건을 추가해 봅시다.
사용할 컬럼은 `weekday_pm_shoulder_p50`이며, 오후 7시 이상 밤 10시 미만의 속도입니다.
3절의 코드 셀부터 다시 실행하면 비교표에 새 행이 생깁니다.

오후 6시와 저녁 8시의 통행시간을 비교하고, 도로 연결이 같아도 결과가 달라지는 이유를 설명해 봅시다.
관측이 없는 도로에는 어떤 속도가 들어갔는지도 함께 적습니다.

산출물: 시간대별 비교표 한 개와 해석 3~4줄.